# Linear-Elastic Plate with Hole — Provenance Plots

This notebook fetches benchmark provenance data from RoHub and visualises how the simulation results depend on the mesh element size and the polynomial degree of the isoparametric elements.

Each series in the plots corresponds to one combination of simulation tool and element degree. The x-axis uses a logarithmic scale; error plots additionally use a logarithmic y-axis so that convergence rates appear as straight lines.

## Setup

Import the helper functions from the `provenance` package.

In [ ]:
import os
import sys

provenance_path = os.path.abspath(os.path.join("..", "provenance"))

if provenance_path not in sys.path:
    sys.path.append(provenance_path)

from rohub_provenance import (
    configure_rohub,
    find_annotated_ro_uuids,
    find_named_graphs_for_uuids,
    query_metric_data_from_named_graphs,
    ANNOTATION_CONFIG,
)
from plot_metrics import select_plot_columns, plot_provenance_graph

## Configuration

In [ ]:
BENCHMARK_NAME = "linear-elastic-plate-with-hole"
USE_PRODUCTION_ROHUB = True  # set to False to use the development RoHub instance

## Connect to RoHub and find RO-Crates

Configure the RoHub client and look up all RO-Crates that are annotated with this benchmark and the current branch of the code repository.
Each RO-Crate corresponds to one simulation run and contains the recorded parameters and metrics.

In [ ]:
configure_rohub(use_production_rohub=USE_PRODUCTION_ROHUB)

code_repository_url = ANNOTATION_CONFIG["main_branch_url"]
print(f"Searching for RO-Crates annotated with: {code_repository_url}")

uuids = find_annotated_ro_uuids(BENCHMARK_NAME, code_repository_url=code_repository_url)
print(f"Found {len(uuids)} RO-Crate(s)")

## Resolve SPARQL named graphs

Each RO-Crate is stored in a dedicated named graph in the RoHub triple store.
Resolving the UUIDs to graph URIs allows the subsequent data query to target only the relevant graphs.

In [ ]:
named_graphs = find_named_graphs_for_uuids(uuids, use_production_rohub=USE_PRODUCTION_ROHUB)

print(f"Resolved {len(named_graphs)} named graph(s):")
for uuid, graph_uri in named_graphs.items():
    print(f"  {uuid}\n    -> {graph_uri}")

## Fetch simulation results

Query the named graphs for the simulation parameters and result metrics.
The result is a pandas DataFrame where each row is one simulation run.

In [ ]:
parameters = ["element_size", "isoparametric_element_degree"]
metrics = ["max_von_mises_stress", "max_displacement_error", "l2_error_displacement"]

data = query_metric_data_from_named_graphs(
    parameters=parameters,
    metrics=metrics,
    named_graphs=list(named_graphs.values()),
)

data

## Maximum von Mises Stress

Plot the maximum von Mises stress against the element size. A finer mesh generally resolves stress concentrations more accurately, so the value is expected to increase and converge towards the analytical solution as the element size decreases.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["max_von_mises_stress"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="Max von Mises Stress",
    title="Max von Mises Stress vs Element Size",
)

## Maximum Displacement Error

Plot the maximum displacement error against the element size on a log–log scale. For a well-converging method the error decreases as a power of the element size, appearing as a straight line whose slope equals the convergence rate.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["max_displacement_error"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="Max Displacement Error",
    title="Max Displacement Error vs Element Size",
    log_y=True,
)

## L2 Displacement Error

Plot the L2 norm of the displacement error against the element size on a log–log scale. The L2 error averages the error over the whole domain.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["element_size", "isoparametric_element_degree"],
    metrics=["l2_error_displacement"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Element Size",
    y_axis_label="L2 Error Displacement",
    title="L2 Error Displacement vs Element Size",
    log_y=True,
)